In [1]:
# General imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy import integrate
from copy import deepcopy
from astropy.io import fits

# Import euclidlib for reading the Euclid data
import euclidlib as el

# Import cloelib for cosmology and theoretical predictions
from cloelib.cosmology.camb_cosmology import CAMBBackground
from cloelib.cosmology.HMcode2020Emu_cosmology import HMemuLinearPerturbations, HMemuNonLinearPerturbations

# Import cloelike for likelihoods
from cloelike.EuclidLikelihood_WL_Cls import EuclidLikelihood_WL_Cls
from cloelike.EuclidLikelihood_GC_Cls import EuclidLikelihood_GC_Cls
from cloelike.EuclidLikelihood_2x2pt_Cls import EuclidLikelihood_2x2pt_Cls
from cloelike.EuclidLikelihood_3x2pt_Cls import EuclidLikelihood_3x2pt_Cls

/Users/guadalupe.canasherr/miniforge3/envs/cloe-org/lib/python3.10/site-packages/HMcode2020Emu
Loading linear emulator...


Linear emulator loaded in memory.
Loading nonlinear emulator...
Non-linear emulator loaded in memory.
Loading linear emulator...
Baryonic boost emulator loaded in memory.
Loading sigma8 emulator...
Linear emulator loaded in memory.


# Download synthetic data

At the moment, we are working on providing homogeneous synthetic data vectors, mixing matrices and covariance matrices that are readable by euclidlib and consistent with the format of the Euclid Science Ground Segment LE3 files. This is still work in progress.

In [2]:
# Execute this cell to download the data for testing the likelihood calculations
data_downloaded = False

if data_downloaded == False:
    
    import requests
    
    # URLs of the files to be downloaded
    urls = {
        'nz_example.fits': 'https://zenodo.org/records/15092862/files/nz_example.fits',
        'cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy': 'https://zenodo.org/records/15496892/files/cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy',
        'mixmat_identity_5000_binned.fits': 'https://zenodo.org/records/15496892/files/mixmat_identity_5000_binned.fits',
        'synth_cells_5000_binned.fits': 'https://zenodo.org/records/15496892/files/synth_cells_5000_binned.fits',
    }
    
    # Function to download a file
    def download_file(url, filename):
        response = requests.get(url)
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(response.content)
            print(f'{filename} downloaded successfully')
        else:
            print(f'Failed to download {filename}. Status code: {response.status_code}')
    
    # Download all files
    for filename, url in urls.items():
        download_file(url, filename)

nz_example.fits downloaded successfully
cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy downloaded successfully
mixmat_identity_5000_binned.fits downloaded successfully
synth_cells_5000_binned.fits downloaded successfully


# Read the data and normalise n(z)

You will need to have installed euclidlib.

In [3]:
# Get n(z)
z_nz, nz_heracles = el.photo.redshift_distributions('nz_example.fits')

# Normalize and resample n(z) for both position and shear
myz = np.linspace(1e-4, 3.0, 100)
def normalize_and_resample(nz_dict, z_grid, z_target):
    nz_array = np.vstack([nz / integrate.trapezoid(nz, z_grid) for nz in nz_dict.values()])
    return np.array([np.interp(z_target, z_grid, nz) for nz in nz_array])

my_dndz_pos_norm = normalize_and_resample(nz_heracles, z_nz, myz)
my_dndz_she_norm = normalize_and_resample(nz_heracles, z_nz, myz)


In [4]:
# We read with euclidlib v2025.2 (v2025.1 is also compatible)
cells_data = el.photo.angular_power_spectra('synth_cells_5000_binned.fits')
mixmat = el.photo.mixing_matrices('mixmat_identity_5000_binned.fits')

# This matrix copies the format of seaborne, that produces a full 3x2pt matrix
full_cov=np.load('cov_Gauss_3x2pt_2D_probe_zpair_ell_2500deg2_ellmax5000_Bmode_copy.npy')

# Cutting the covariance matrix for each probe
# This should be easier with the new version of euclidlib
cov_WL = full_cov[:(21+21)*32, :(21+21)*32]
cov_GC = full_cov[(21+21+36)*32:, (21+21+36)*32:]
cov_2x2pt = full_cov[(21+21)*32:, (21+21)*32:]

# Prepare data dictionary

In [5]:
def build_data(ell_key, cov, include_pos=False, include_she=False):
    data = {
        'cells': cells_data,
        'ells': cells_data[ell_key].ell,
        'z_arr': myz,
        'cov': cov,
        'mixmat': mixmat,
    }
    if include_pos:
        data['dndz_pos'] = my_dndz_pos_norm
    if include_she:
        data['dndz_she'] = my_dndz_she_norm
    return data

def build_settings():
    scale_cuts = {key: [10, 1500] for key in cells_data}
    for key in cells_data:
        if key[:2] == ('SHE', 'SHE'):
            scale_cuts[key] = [scale_cuts[key], [0, 0]]
    return {'n_ell_bins': 32, 'scale_cuts': scale_cuts}

# Build each dataset with correct dndz components
data_WL     = build_data(('SHE', 'SHE', 1, 1), cov_WL,     include_she=True)
data_GC     = build_data(('POS', 'POS', 1, 1), cov_GC,     include_pos=True)
data_2x2pt  = build_data(('POS', 'POS', 1, 1), cov_2x2pt,  include_pos=True, include_she=True)
data_3x2pt  = build_data(('POS', 'POS', 1, 1), full_cov,   include_pos=True, include_she=True)

# Settings (same for all, built from cells_data structure)
settings_WL     = build_settings()
settings_GC     = build_settings()
settings_2x2pt  = build_settings()
settings_3x2pt  = build_settings()



# Likelihood instances

In [6]:
import time

likelihood_classes = {
    'WL': EuclidLikelihood_WL_Cls,
    'GC': EuclidLikelihood_GC_Cls,
    '2x2pt': EuclidLikelihood_2x2pt_Cls,
    '3x2pt': EuclidLikelihood_3x2pt_Cls,
}

data_dicts = {
    'WL': (data_WL, settings_WL),
    'GC': (data_GC, settings_GC),
    '2x2pt': (data_2x2pt, settings_2x2pt),
    '3x2pt': (data_3x2pt, settings_3x2pt),
}

like_tests = {}
loglike_results = {}

print("🔍 Timing log-likelihood initialisation (this only happens once) and displaying results:")

for label in ['WL', 'GC', '2x2pt', '3x2pt']:
    print(f"\n📊 {label} loglike:")
    data, settings = data_dicts[label]
    cls = likelihood_classes[label]

    start = time.time()
    like_tests[label] = cls(
        data=data,
        settings=settings,
        Background=CAMBBackground,
        LinPerturbations=HMemuLinearPerturbations,
        NonLinPerturbations=HMemuNonLinearPerturbations,
    )
    end = time.time()

    print(f"⏱️ Time elapsed ({label}): {end - start:.3f} seconds")


🔍 Timing log-likelihood initialisation (this only happens once) and displaying results:

📊 WL loglike:
⏱️ Time elapsed (WL): 0.037 seconds

📊 GC loglike:
⏱️ Time elapsed (GC): 0.026 seconds

📊 2x2pt loglike:
⏱️ Time elapsed (2x2pt): 0.101 seconds

📊 3x2pt loglike:
⏱️ Time elapsed (3x2pt): 0.166 seconds


# Likelihood evaluations

In [7]:
# These are the default parameters used to generate the synthetic data, therefore, the log-likelihood should be close to zero.
default_pars = {'H0':67,'Omega_cdm0':0.27,'Omega_b0':0.049,'ns':0.96,'As':2.1e-9,
                'w0':-1,'wa':0, 'Omega_k0':0,'mnu':0.0,'gamma_MG':0.545,
                'log10TAGN': 7.75,
                'AIA':0.16, 'EtaIA':1.66,
                'b1_photo_poly0': 1.33291, 'b1_photo_poly1': -0.72414,
                'b1_photo_poly2': 1.0183, 'b1_photo_poly3': -0.14913,
                'magnification_bias_1': 0.0, 'magnification_bias_2': 0.0,
                'magnification_bias_3': 0.0, 'magnification_bias_4': 0.0,
                'magnification_bias_5': 0.0, 'magnification_bias_6': 0.0,
                'dz_pos_1': 0.0, 'dz_pos_2': 0.0,
                'dz_pos_3': 0.0, 'dz_pos_4': 0.0,
                'dz_pos_5': 0.0, 'dz_pos_6': 0.0,
                'multiplicative_bias_1': 0.0, 'multiplicative_bias_2': 0.0,
                'multiplicative_bias_3': 0.0, 'multiplicative_bias_4': 0.0,
                'multiplicative_bias_5': 0.0, 'multiplicative_bias_6': 0.0,
                'dz_shear_1': 0.0, 'dz_shear_2': 0.0,
                'dz_shear_3': 0.0, 'dz_shear_4': 0.0,
                'dz_shear_5': 0.0, 'dz_shear_6': 0.0}

In [8]:
# Dictionary of pretty names for output
labels_pretty = {
    'WL': "Weak Lensing (WL)",
    'GC': "Photometric Angular Galaxy Clustering (GC)",
    '2x2pt': "2x2pt (GC + GGL)",
    '3x2pt': "3x2pt"
}

print("⏱️ Timing log-likelihood evaluations and displaying results:")

loglike_results = {}

for label in ['WL', 'GC', '2x2pt', '3x2pt']:
    print(f"\n📊 {labels_pretty[label]} loglike:")
    start = time.time()
    result = like_tests[label].loglike(default_pars)
    elapsed = time.time() - start
    loglike_results[label] = result
    print(f"✅ Returned value ({label}): {result}")
    print(f"⏱️ Time elapsed ({label}): {elapsed:.3f} seconds")


⏱️ Timing log-likelihood evaluations and displaying results:

📊 Weak Lensing (WL) loglike:
✅ Returned value (WL): -1.7563844085995965e-09
⏱️ Time elapsed (WL): 18.444 seconds

📊 Photometric Angular Galaxy Clustering (GC) loglike:
✅ Returned value (GC): -2.456348262212602e-09
⏱️ Time elapsed (GC): 0.968 seconds

📊 2x2pt (GC + GGL) loglike:
✅ Returned value (2x2pt): -5.651043116169738e-09
⏱️ Time elapsed (2x2pt): 0.380 seconds

📊 3x2pt loglike:
✅ Returned value (3x2pt): -9.846737316651602e-09
⏱️ Time elapsed (3x2pt): 0.387 seconds


In [9]:
# Test that 3x2pt ≈ 2x2pt + WL (within numerical tolerance)
diff = loglike_results['3x2pt'] - (loglike_results['2x2pt'] + loglike_results['WL'])
tol = 1e-6  # adjust tolerance as needed

if np.abs(diff) < tol:
    print(f"\n✅ Consistency test passed: 3x2pt ≈ 2x2pt + WL (diff = {diff:.2e})")
else:
    print(f"\n⚠️ Consistency test FAILED: 3x2pt differs from 2x2pt + WL by {diff:.2e}")


✅ Consistency test passed: 3x2pt ≈ 2x2pt + WL (diff = -2.44e-09)
